In [ ]:
# Medora Fine-Tuning — Kaggle Notebook
# Run on Kaggle with GPU T4 x2 (free tier)
# Trains Gemma 4 E4B on the Medora medication counseling dataset

In [ ]:
!pip install unsloth

In [ ]:
from unsloth import FastModel
import torch

model, tokenizer = FastModel.from_pretrained(
model_name="unsloth/gemma-4-E2B-it",
max_seq_length=2048,
load_in_4bit=True,
full_finetuning=False,
)

In [ ]:
model = FastModel.get_peft_model(
model,
r=32,
target_modules=[
"q_proj", "k_proj", "v_proj", "o_proj",
"gate_proj", "up_proj", "down_proj",
],
lora_alpha=64,
lora_dropout=0,
use_gradient_checkpointing="unsloth",
random_state=42,
)

In [ ]:
from datasets import load_dataset

# v2 dataset: each system prompt now ends with a MEDICATION REFERENCE block
# matching the production prompt shape from service._build_medication_reference.
dataset = load_dataset(
    "json",
    data_files="/kaggle/input/datasets/wallfou/medora-training-data3/medora_train_v2.jsonl",
    split="train",
)

# Validating dataset shape
print(f"Total training examples: {len(dataset)}")
for i, row in enumerate(dataset):
    msgs = row.get("messages", [])
    if len(msgs) < 2:
        print(f"WARNING: Row {i} has only {len(msgs)} messages")
    if msgs[0]["role"] != "system":
        print(f"WARNING: Row {i} first message is not system role")

# Confirm reference block is present so we don't accidentally train on v2 data
sample_sys = dataset[0]["messages"][0]["content"]
assert "Background facts about the patient's medications" in sample_sys, (
    "Reference block missing from system prompt -- check the data file"
)
print(f"Sample question: {dataset[0]['messages'][1]['content'][:80]}...")
print(f"Sample answer: {dataset[0]['messages'][2]['content'][:80]}...")

In [ ]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template="gemma-4",
)

def format_example(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

dataset = dataset.map(format_example)

# Split 90/10 for train/eval to detect overfitting
split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
eval_dataset = split["test"]
print(f"Train: {len(train_dataset)}, Eval: {len(eval_dataset)}")

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=1024,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=6,
        learning_rate=1e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        eval_strategy="steps",
        eval_steps=20,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
        report_to="none",
    ),
)

# Detect which turn tokens the template actually uses
sample_text = train_dataset[0]["text"]
if "<start_of_turn>user" in sample_text:
    inst_token = "<start_of_turn>user\n"
    resp_token = "<start_of_turn>model\n"
    print("Detected: <start_of_turn> style tokens")
elif "<|turn>user" in sample_text:
    inst_token = "<|turn>user\n"
    resp_token = "<|turn>model\n"
    print("Detected: <|turn> style tokens")
else:
    raise ValueError(
        f"Could not detect turn tokens in formatted text. "
        f"First 500 chars:\n{sample_text[:500]}"
    )

# Apply response only training
trainer = train_on_responses_only(
    trainer,
    instruction_part=inst_token,
    response_part=resp_token,
)

# Verify masking actually worked
sample = trainer.train_dataset[0]
labels = sample["labels"]
masked = sum(1 for l in labels if l == -100)
total = len(labels)
pct = masked / total
print(f"Masked {masked}/{total} tokens ({pct:.0%})")
if pct < 0.2 or pct > 0.95:
    raise ValueError(
        f"Masking looks wrong ({pct:.0%})."
    )
print("Response only masking verified.")

print("\nStarting training...")
trainer_stats = trainer.train()
print(f"Training time: {trainer_stats.metrics['train_runtime']:.0f}s")
print(f"Final train loss: {trainer_stats.metrics['train_loss']:.4f}")

In [ ]:
from transformers import AutoTokenizer
text_tokenizer = AutoTokenizer.from_pretrained("unsloth/gemma-4-E2B-it")

def run_test(messages, max_tokens=300):
    text = text_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = text_tokenizer(text, return_tensors="pt").to("cuda")
    from transformers import TextStreamer
    streamer = TextStreamer(text_tokenizer, skip_prompt=True)
    _ = model.generate(
        **inputs,
        streamer=streamer,
        max_new_tokens=max_tokens,
        temperature=0.3,
        top_p=0.9,
    )

# Persona + reference block mirroring the production prompt shape so the
# smoke tests evaluate the model on inputs it actually saw during training.
PERSONA = (
    "You are Medora, a warm and knowledgeable medication safety assistant. "
    "The patient takes these medications: warfarin. "
    "Use short, simple sentences. Explain medical terms in plain language. "
    "Share standard patient education information freely. "
    "Defer diagnosis and dosage decisions to doctors. "
    "Never repeat the same disclaimer twice in a row."
)
REFERENCE_WARFARIN = (
    "Background facts about the patient's medications. Weave in only what "
    "directly answers the patient's question; do not list every field unless "
    "asked. Stay in your normal warm, conversational voice. For drugs not "
    "listed here, use your general training knowledge.\n\n"
    "WARFARIN\n"
    "  Patient's dose: 5mg daily\n"
    "  Common side effects: Easier bruising and bleeding (gums, nose, cuts). "
    "Serious signs: blood in urine or stool, severe headache, or heavy "
    "persistent bleeding."
)
SYS_WARFARIN = f"{PERSONA}\n\n{REFERENCE_WARFARIN}"

print("\nTest 1: Side effects of warfarin: ")
run_test([
    {"role": "system", "content": SYS_WARFARIN},
    {"role": "user", "content": "What are the side effects of warfarin?"},
])

print("\nTest 2: Off-topic handling: ")
run_test([
    {"role": "system", "content": SYS_WARFARIN},
    {"role": "user", "content": "How do I use recursion to sort an array?"},
], max_tokens=100)

print("\nTest 3: Empathy: ")
run_test([
    {"role": "system", "content": SYS_WARFARIN},
    {"role": "user", "content": "I'm worried I'm addicted to oxycodone. What should I do?"},
])

# Regression check for the original bug: an elderly patient asking about
# warfarin side effects should NOT receive pregnancy/teratogen warnings.
print("\nTest 4 (regression): Elderly patient + warfarin side effects: ")
run_test([
    {"role": "system", "content": SYS_WARFARIN.replace(
        "warfarin.",
        "warfarin. The patient is 78 years old.",
    )},
    {"role": "user", "content": "What are the side effects of warfarin?"},
])

In [ ]:
model.save_pretrained("medora-lora")
tokenizer.save_pretrained("medora-lora")
print("LoRA adapters saved to medora-lora/")

In [ ]:
!pip install -U huggingface_hub

import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

login(token=hf_token)

hf_username = "Wallfou"
repo_name = "medora-gemma-4-lora"
repo_id = f"{hf_username}/{repo_name}"

model.push_to_hub(repo_id, token=hf_token)
tokenizer.push_to_hub(repo_id, token=hf_token)